## Imports and model

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, set_seed
import gc
import random
import numpy as np

model_id = "unsloth/Llama-3.2-1B-Instruct"
query = "why is 42 a special number?"

messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": query},
]

/home/hx/hx/vllm_practice/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Generating using pipeline

Hugging face transformers provides pipeline function, which processes the input and generates ouput, based on the task(text-generation in this case). This is the easiest way of generating outputs from input prompt with a model, where tokenization, model loading and decoding/generation is performed by the in-built function

In [2]:
# from transformers import pipeline
pipe = pipeline(
    "text-generation",
    model=model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

outputs = pipe(
    messages,
)
print(outputs[0]["generated_text"][-1]['content'])

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 146/146 [00:00<00:00, 402.93it/s]
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


You're referring to the famous "42" that's often associated with Douglas Adams' science fiction series "The Hitchhiker's Guide to the Galaxy." In the book, the number 42 is considered special because it's the "Answer to the Ultimate Question of Life, the Universe, and Everything."

According to the book, a supercomputer named Deep Thought was asked to find the answer to this question, and after thinking for 7.5 million years, it finally revealed that the answer was 42. However, the characters in the book realize that they don't actually know what the ultimate question is, so the number 42 remains a mystery.

The significance of 42 has since become a cultural phenomenon, with many people believing that it holds the key to unlocking the secrets of the universe. However, Douglas Adams himself said that he made 42 up because he was trying to come up with a number that was "fun" and "exciting," rather than a real mathematical solution to a profound question.

Despite the lack of a definitiv

The above output might differ because seed isn't set for it's generation. But we set the seed for next generation and should expect reproducible results.

In [3]:
# Run these to remove the model from GPU VRAM. In my case I have 6GB of VRAM, hence removing the earlier loaded model from GPU

del pipe
gc.collect()
torch.cuda.empty_cache()

## Generating using tokenizer and model loading
Hugging Face transformers provides us classes for model loading and tokenization. We use these classes to load the model, process the input prompt and use the "generate" function provided by the model class to obtain outputs

Note: restart python kernel

In [2]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",   # Automatically selects the correct precision
    device_map="auto"     # Automatically allocates model layers across available hardware (GPU/CPU)
)

Loading weights: 100%|██████████| 146/146 [00:00<00:00, 411.65it/s]


In [5]:
# Converting the input messages from above into chat template; This is implicitly done by the pipeline above.
input_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(input_prompt)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 06 Sep 2026

You are a helpful assistant<|eot_id|><|start_header_id|>user<|end_header_id|>

why is 42 a special number?<|eot_id|><|start_header_id|>assistant<|end_header_id|>




In [6]:
inputs = tokenizer(input_prompt, return_tensors="pt").to(model.device)
input_len = inputs['input_ids'].shape[1]


In [7]:
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        do_sample=True,
        temperature=0.6,
        top_p=0.9,
    )

In [8]:
generated_text = tokenizer.decode(output_ids[0, input_len:], skip_special_tokens=True) #output_ids contain input_ids as well, so we slice them out with output_ids[0, input_len:]
print(generated_text)

You're referring to the famous "42" from Douglas Adams' science fiction series "The Hitchhiker's Guide to the Galaxy." In the book, the number 42 is considered special because it is the answer to the ultimate question of life, the universe, and everything, as mentioned in the opening lines of the book.

In the story, the supercomputer Deep Thought, which has been asked to calculate the answer to the ultimate question, takes 7.5 million years to arrive at the answer, which is then revealed to be 42. This leads to a humorous and satirical exploration of the nature of the question itself, as well as the search for meaning and the futility of seeking definitive answers to complex questions.

In the context of the book, 42 is not just a number; it's a symbol of the absurdity and complexity of the universe, and a commentary on the search for meaning and purpose in a seemingly meaningless world.

However, in a more practical sense, 42 has become a cultural phenomenon, often used as a meme and

## Manual decoding

We implement the naive decode loop manually using greedy approach, generating and attaching one token to the input at a time

### Naive way

#### Obtaining next token

In [9]:
inputs = tokenizer(input_prompt, return_tensors="pt").to(model.device)
inputs_raw = inputs['input_ids']
input_len = inputs_raw.shape
print(f"Shape of inputs: {inputs['input_ids'].shape}")

Shape of inputs: torch.Size([1, 49])


In [10]:
#Before implementing decoding, let's first check what is the shape of the output of the model
model.eval()
with torch.no_grad():
    outputs = model(inputs_raw)
print(outputs.logits.shape)


torch.Size([1, 49, 128256])


The ouput contains logits which is of shape (n_batch, n_time_step, logit_value)

In [11]:
logits = outputs.logits[:,-1,:] # gives the logits of final layer
output_token_id = logits[0].argmax() # following greedy decoding, we pick up the logit with max value

print(f"The next prdicted token is: {tokenizer.decode(output_token_id)}")

The next prdicted token is: You


#### Implementing decoding loop based on max_len

In [12]:
#we add the next predicted token id from prev step and forward pass it multiple times to obtain the generated content

max_len = 100
inputs = tokenizer(input_prompt, return_tensors="pt").to(model.device)
inputs_raw = inputs['input_ids']

input_len = inputs_raw.shape
input_len = inputs_raw.shape[-1]
output_ids = inputs_raw
model.eval()
with torch.no_grad():
    for _ in range(max_len):
        outputs = model(output_ids)
        logits = outputs.logits[:,-1,:]
        next_output_id = logits.argmax(keepdim=True)
        output_ids = torch.cat([output_ids, next_output_id], -1)


In [13]:
generated_ids = output_ids[:, input_len:]
print(tokenizer.decode(generated_ids)[0])

You're referring to the famous "42" that appears in Douglas Adams' science fiction series "The Hitchhiker's Guide to the Galaxy." In the book, the number 42 is often considered special because it is the answer to the Ultimate Question of Life, the Universe, and Everything, which is the central mystery of the series.

The book's author, Douglas Adams, intentionally left the answer blank, leaving it up to the reader's imagination to fill in the number. This has led to


#### Implementing decoding loop based on EOS token

In [ ]:
eos_ids = [128001,128008,128009]

#we add the next predicted token id from prev step and forward pass it multiple times to obtain the generated content
inputs = tokenizer(input_prompt, return_tensors="pt").to(model.device)
inputs_raw = inputs['input_ids']

input_len = inputs_raw.shape[-1]
output_ids = inputs_raw

model.eval()
with torch.no_grad():
    for _ in range(10000): # setting max_len to 10000 to avoid infinite loop in case eos token is not predicted
        outputs = model(output_ids)
        logits = outputs.logits[:,-1,:]
        next_output_id = logits.argmax(keepdim=True)
        if next_output_id.item() in eos_ids:
            break
        output_ids = torch.cat([output_ids, next_output_id], -1)

In [22]:
generated_ids = output_ids[:, input_len:]
print(tokenizer.decode(generated_ids)[0])

You're referring to the famous "42" that appears in Douglas Adams' science fiction series "The Hitchhiker's Guide to the Galaxy." In the book, the number 42 is often considered special because it is the answer to the Ultimate Question of Life, the Universe, and Everything, which is the central mystery of the series.

The book's author, Douglas Adams, intentionally left the answer blank, leaving it up to the reader's imagination to fill in the number. This has led to much speculation and debate over the years, with some people trying to solve the mystery and others dismissing it as a joke.

However, the true answer has never been officially revealed, and it's become a kind of cultural phenomenon. Some people have even created their own theories and interpretations of the number 42, ranging from mathematical solutions to philosophical and scientific explanations.

In reality, the number 42 is simply a placeholder number that was chosen by Adams to represent the answer to the Ultimate Que

#### Utilizing KV Cache

In [23]:
print(outputs.keys())

odict_keys(['logits', 'past_key_values'])


The output of the model has 2 keys - logits and past_key_values. past_key_values has cache of KV layers which can be used to speed up the decoding process.

The above implemented decoding loop sends all the token IDs to the model at each step and processes on all of them, instead we can use the past_key_values to send only the last generated token and use KV values from the cache

In [ ]:
eos_ids = [128001,128008,128009]

#we add the next predicted token id from prev step and forward pass it multiple times to obtain the generated content
inputs = tokenizer(input_prompt, return_tensors="pt").to(model.device)
inputs_raw = inputs['input_ids']
inputs_attn = inputs['attention_mask']
past = None

input_len = inputs_raw.shape[-1]
output_ids = inputs_raw
output_attn = inputs_attn

model.eval()
with torch.no_grad():
    for _ in range(10000): # setting max_len to 10000 to avoid infinite loop in case eos token is not predicted
        outputs = model(input_ids = output_ids[:, -1:] if past else output_ids,
                        attention_mask = output_attn,
                        past_key_values = past,
                        use_cache = True)
        logits = outputs.logits[:,-1,:]
        past = outputs.past_key_values
        next_output_id = logits.argmax(keepdim=True)
        if next_output_id.item() in eos_ids:
            break
        output_ids = torch.cat([output_ids, next_output_id], -1)
        output_attn = torch.cat([output_attn, torch.ones_like(next_output_id)], -1)

In [35]:
generated_ids = output_ids[:, input_len:]
print(tokenizer.decode(generated_ids)[0])

You're referring to the famous "42" that appears in Douglas Adams' science fiction series "The Hitchhiker's Guide to the Galaxy." In the book, the number 42 is often considered special because it is the answer to the Ultimate Question of Life, the Universe, and Everything, which is the central mystery of the series.

The book's author, Douglas Adams, intentionally left the answer blank, leaving it up to the reader's imagination to figure out what the question is. This has led to much speculation and debate over the years, with some people trying to come up with their own answers.

In reality, the number 42 has become a cultural phenomenon, symbolizing the search for meaning and the quest for answers to life's big questions. It has also been used as a meme and a symbol of geek culture, representing the idea that there is no one definitive answer to the mysteries of the universe.

So, while 42 may not have a definitive answer, it has become a beloved and iconic number that represents the

In my initial run, the decode loop without the KV Cache took 8.8s, whereas the one with KV cache took 2.7s. Speedup of ~3.25x

## Model Math

In [37]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048, padding_idx=128004)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,)

In [43]:
model_params = model.num_parameters()
print(f"{model_params/1e9:.3f}B") 


1.236B


We load the model in bf16, so each parameter takes 2 bytes. Total VRAM for loading the model = no. params * 2 bytes

In [44]:
vram_usage_exp = model_params * 2
print(f"{vram_usage_exp/1e9:.3f}GB") 

2.472GB


Actual VRAM usage from nvidia-smi upon model loading: 2.439GB

KV Cache Math:
This Llama model has head_dim=64 (from config), and 8 heads for K & V

The KV cache VRAM usage, therefore is, <br>
2 bytes(bf16) * L * KV_heads (i.e., K + V) * head_dim * seq_len * batch <br>
2 * 16 * 16 * 64 * seq_len * batch

In [6]:
#for 100 tokens and batch size 1, the KV cache occupies the following VRAM
print(f"{(2 * 16 * (8 + 8) * 64 * 100 * 1) / 1e6:0.3f} MB")


3.277 MB
